# Suite REF — References: one format for objects, relation entries, frames and parameters

A reference designates what a request or a response points at ([CAP-130: references to objects, entries and frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-130.md#decided)):
`References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object), never a counted one, which nothing names (object identity 40 and 55 in [CAP-118: singletons, owned and counted objects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-118.md#decided));
`References.OfEntry`, a relation entry, by its relation's path and all its links and properties; `References.OfFrame`,
a frame, each symbol's object and each variable's value, and what it witnesses, objects' property values
(`References.OfProperties`) and relation entries, each by its objects' references, and the errors annotated on it
(`References.OfError`, [CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)). `References.of(store, x)`
gives one, a value of the store's, written and read as any value is. A reference is resolved within a transaction, as
every read of a store's data is ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md)): a map whose symbol is given an object's reference
binds what it designates, and gives no frame where the store holds none; a frame's reference is resolved by a map
whose symbols are given the references it binds, and an entry's is matched by a pattern over the objects it links. `References.OfParameter`, a parameter, holds its name and
nothing else; a binding in the scope that encloses it resolves it, not a store.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Paths, Plain, References, Transactions
from support import raises


def crm(txn):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
    about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject, "About": S.OfRelation})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]).adjacencies({"trio": lambda t: t.relation([S.OfRelation(lambda r: r.links([  # no name of its own
                lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book, Note = (store.registered(name) for name in ("crm.Person", "crm.Book", "crm.Note"))
frames = store.transaction(lambda t: t.uid("people").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"book": Book, "ada": Person, "bob": Person, "cy": Person}).spec([
        lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob),
                                   "cy": lambda e: e.contact(_.cy)}),
        lambda _: _.ada.name("Ada").knows([lambda e: e.known(_.bob).since(2020), lambda e: e.known(_.cy).since(2021)]),
        lambda _: _.bob.name("Bob"),
        lambda _: _.cy.name("Cy"),
    ])]))]))
book, ada, bob, cy = (frames[0][name] for name in ("book", "ada", "bob", "cy"))  # each its reference: a frame holds no visitor


def contact(key):
    """The contact the book holds by `key`, within the transaction open: found again by navigating from the singleton."""
    return store.singleton("crm.TheBook").contacts({"key": key})


def notes(t):
    """A note about each person, upserted with the key `notes/1` for the frame that reaches it."""
    return t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"contact": Person})])
        .then([lambda n: n.upsert(lambda u: u
        .key("notes/1").exists([lambda p: p.symbols({"contact": Person, "note": Note}).spec([
            lambda _: _.note.text("hi").about([_.contact])])]))]))])


def plain(reference):
    return Plain.ToPlain(store)(reference.as_schema(), reference)


def plains(frames):
    """Frames given back, each symbol's reference in its plain form, a schema object as itself, a counted object, which
    the frame refuses, as "counted": what tells frames apart, a frame holding references made afresh in each transaction
    ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md))."""
    def one(frame, name):
        try:
            held = frame[name]
        except LookupError:
            return "counted"
        return plain(held) if store.name_of(held.as_schema()) == "References.OfObject" else held
    return [{name: one(frame, name) for name in frame} for frame in frames]


def path(singleton, *steps):
    """A path's plain form: from a singleton through entries, each (i, o, keys)."""
    nested = {"target": {}}
    for i, o, keys in reversed(steps):
        nested = {"entry": {"i": i, "o": o, "properties": keys, "next": nested}}
    return {"singleton": singleton, "next": nested}


def bound(held, references):
    """The frames of a map whose symbols are each given a reference: within the transaction that runs it, each symbol
    binds what its reference designates, so that the map gives one frame, or none where `held` holds nothing one
    designates ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md))."""
    return held.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols(dict(references))]))]))


def designated(held, reference):
    """What an object's reference designates in `held`, by a map whose one symbol is given it; None where it holds none."""
    frames = bound(held, {"x": reference})
    return frames[0]["x"] if frames else None


def framed(held, reference):
    """The frames of a map whose symbols are given the references a frame's reference binds, read off its plain form, as
    a program it is handed to reads them: the frame again, or none."""
    OfObject = held.registered("References.OfObject")
    binds = Plain.ToPlain(held)(reference.as_schema(), reference).get("binds", {})
    return bound(held, {name: Plain.FromPlain(held)(OfObject, given) for name, given in binds.items()})

<a id="REF-01"></a>
## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
OfObject = store.registered("References.OfObject")
OfFrame, OfEntry = store.registered("References.OfFrame"), store.registered("References.OfEntry")
assert list(OfObject.properties) == ["path"] and OfObject.properties["path"].schema is store.registered("Paths.OfRoot")
OfProperties = store.registered("References.OfProperties")
assert list(OfFrame.properties) == ["binds", "variables", "assigns", "entries", "errors"]
assert OfFrame.properties["binds"].schema.element is OfObject and OfFrame.properties["assigns"].schema.element is OfProperties
assert OfFrame.properties["entries"].schema.element is OfEntry and list(OfProperties.properties) == ["object", "properties"]
assert list(OfEntry.properties) == ["relation", "links", "properties"]
assert OfEntry.properties["relation"].schema is store.registered("Paths.OfRoot")
assert OfEntry.properties["links"].schema.element is OfObject
OfError = store.registered("References.OfError")
assert OfFrame.properties["errors"].schema.element is OfError and list(OfError.properties) == ["subterm", "kind", "message"]
assert not any(schema.ref for schema in (OfObject, OfFrame, OfProperties, OfEntry, OfError))  # value objects, as a path is
other = B.Store()
assert other.registered("References.OfObject") is not OfObject  # each store's its own
names = {name: store.registered(name) for name in store.names()}
store.transaction(References.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == names

<a id="REF-02"></a>
## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each designated by a map whose symbol is given it

In [ ]:
assert plain(References.of(store, book)) == {"path": path("crm.TheBook")}
assert plain(References.of(store, ada)) == {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}
assert plain(References.of(store, Person)) == {"path": path(".", ("schemas", "", {"name": "crm"}),
                                                              ("schemas", "", {"name": "Person"}))}
for target in (book, ada):  # by a map whose symbol is given it: what it designates, by its reference again
    assert plain(designated(store, References.of(store, target))) == plain(target)
assert designated(store, References.of(store, Person)) is Person  # a schema object: itself
built = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "bob"}).next(lambda n: n.target(lambda t: t))))))
assert plain(designated(store, built)) == plain(bob)  # written through the builders' fluent calls

<a id="REF-03"></a>
## REF-03 · A counted object has no reference, nor a frame that binds one: an upsert's, which the executor alone finds again

In [ ]:
made = store.transaction(notes)
assert plains(made) == [{"contact": plain(x), "note": "counted"} for x in (ada, bob, cy)]
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
    made[0]["note"]  # the frame given back refuses it
with raises(LookupError, match="symbol 'note' of the frame: a counted 'crm.Note' has no reference"):
    References.of(store, made[0])
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"), store.transaction():
    References.of(store, store.crm.Note(lambda n: n.text("hi").about([contact("ada")])))  # read within one: none either
again = store.transaction(notes)  # run again: the executor finds what it made, which nothing outside it names
assert plains(again) == plains(made) and not again[0].assigns  # its notes refused alike: nothing names them

<a id="REF-04"></a>
## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties; refused where it links a counted object

In [ ]:
with store.transaction():
    held = next(iter(store.singleton("crm.TheBook").contacts))  # crm.Holds, keyed by `key`
    assert plain(References.of(store, held)) == {
        "relation": path(".", ("schemas", "", {"name": "crm"}), ("schemas", "", {"name": "Holds"})),
        "links": {"book": {"path": path("crm.TheBook")},
                  "contact": {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}},
        "properties": {"key": {"str": "ada"}}}
    knows = list(contact("ada").knows)
    assert plain(References.of(store, knows[1]))["properties"] == {"since": {"int": 2021}}
with store.transaction():
    store.crm.Person(contact("bob"), lambda b: b.knows(lambda a: a.insert(lambda e: e.known(contact("cy")))))  # since left unset
with store.transaction():
    bare = next(iter(contact("bob").knows))
    assert plain(References.of(store, bare))["properties"] == {}
with raises(LookupError, match="a counted 'crm.Note' has no reference"), store.transaction():  # crm.About links a note
    note = store.crm.Note(lambda n: n.text("hi").about([contact("ada")]))  # a note no frame gives back: made here
    References.of(store, next(iter(note.about)))


def known(x, y, *since):
    """The frames of a map over the references of `x` and `y`, whose spec is the crm.Knows entry linking them, with
    `since` where given: an entry's reference is matched by a pattern over the objects it links, one frame where the
    entry holds, none otherwise."""
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"x": References.of(store, x), "y": References.of(store, y)})
        .spec([lambda _: _.x.knows([lambda e: e.known(_.y).since(*since) if since else e.known(_.y)])])]))]))


assert plains(known(ada, bob, 2020)) == [{"x": plain(ada), "y": plain(bob)}]
assert plains(known(ada, cy, 2021)) == [{"x": plain(ada), "y": plain(cy)}]
assert plains(known(bob, cy)) == [{"x": plain(bob), "y": plain(cy)}]  # since unset: the entry with its links alone
assert known(ada, cy, 1999) == [] and known(cy, ada) == []  # a property no entry has; links no entry has
held = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"b": References.of(store, book), "c": References.of(store, ada)})
    .spec([lambda _: _.b.contacts([lambda e: e.contact(_.c).key("ada")])])]))]))
assert plains(held) == [{"b": plain(book), "c": plain(ada)}]  # held

<a id="REF-05"></a>
## REF-05 · A frame a transaction gave back: each symbol's object; a map whose symbols are given its references gives the frame again

In [ ]:
frame = frames[0]
reference = References.of(store, frame)
assert sorted(plain(reference)["binds"]) == sorted(frame)
assert plain(reference)["binds"]["ada"] == plain(References.of(store, ada))
assert [a["properties"] for a in plain(reference)["assigns"]] == [{"name": {"str": n}} for n in ("Ada", "Bob", "Cy")]  # what it witnesses
assert plains(framed(store, reference)) == plains([frame])  # a map whose symbols are given the references it binds: the frame again
assert plain(References.of(store, {})) == {"binds": {}} and framed(store, References.of(store, {})) == [{}]
assert framed(store, store.References.OfFrame(lambda f: f)) == [{}]  # no binds: the empty frame

<a id="REF-06"></a>
## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
with store.transaction():
    loose = store.crm.Note(lambda n: n.text("by hand"))  # counted, written outright: nothing names it
    with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
        References.of(store, loose)
    with raises(LookupError, match="symbol 'loose' of the frame: a counted 'crm.Note' has no reference"):
        References.of(store, {"loose": loose})
    with raises(TypeError, match="the object belongs to another store"):
        References.of(other, contact("ada"))
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, "x")
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, {"x": "x"})  # a frame binding what is not an object
with store.transaction():
    store.crm.Person(contact("ada"), lambda a: a.trio(lambda t: t.insert(lambda e: e.b(contact("bob")).c(contact("cy")))))
with store.transaction(), raises(LookupError, match="has no path"):  # an entry of a relation with no name of its own
    References.of(store, next(iter(contact("ada").trio)))

<a id="REF-07"></a>
## REF-07 · A reference written and read back as a value; no frame where the store holds nothing it designates

In [ ]:
with store.transaction():  # as the store holds it now: an update of ada's rewrote her entries
    held = References.of(store, next(iter(store.singleton("crm.TheBook").contacts)))  # its reference, taken where read
backs = {}
for name, reference in (("ada", References.of(store, ada)), ("held", held), ("frame", References.of(store, frame))):
    text = JSON.ToJSON(store)(reference.as_schema(), reference)
    backs[name] = JSON.FromJSON(store)(reference.as_schema(), text)
    assert plain(backs[name]) == plain(reference)  # read back the same
assert plain(designated(store, backs["ada"])) == plain(ada) and plains(framed(store, backs["frame"])) == plains([frame])
empty = B.Store()  # its schemas and nothing else
empty.transaction(crm)
for target in (ada, Person):
    reference = References.of(store, target)
    back = JSON.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
                                JSON.ToJSON(store)(reference.as_schema(), reference))
    assert designated(empty, back) is (empty.registered("crm.Person") if target is Person else None)
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton(".").next(lambda n: n.entry(
    lambda e: e.i("schemas").o("x").properties({"name": "crm"}).next(lambda n: n.target(lambda t: t))))))
assert designated(store, nowhere) is None  # a schema object's path whose steps name no entries

<a id="REF-08"></a>
## REF-08 · A frame as a witness: property values an object holds, and relation entries, each by its objects' references

In [ ]:
with store.transaction():  # their paths and an entry's reference, taken where they are read
    at_ada, at_bob = Paths.of(contact("ada")), Paths.of(contact("bob"))
    knowing = plain(References.of(store, next(iter(contact("ada").knows))))
witness = store.References.OfFrame(lambda f: f
    .binds({"ada": lambda o: o.path(at_ada), "bob": lambda o: o.path(at_bob)})
    .assigns([lambda x: x.as_object(lambda p: p.object(lambda o: o.path(at_ada)).properties({"name": "Ada"}))])
    .entries([lambda x: x.as_object(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
        "knower": lambda o: o.path(at_ada), "known": lambda o: o.path(at_bob)}).properties({"since": 2020}))]))
assert plain(witness)["assigns"] == [{"object": plain(References.of(store, ada)), "properties": {"name": {"str": "Ada"}}}]
assert plain(witness)["entries"] == [knowing]  # an OfEntry, as of() gives one
text = JSON.ToJSON(store)(witness.as_schema(), witness)
assert plain(JSON.FromJSON(store)(witness.as_schema(), text)) == plain(witness)  # carried, and read back the same
assert plains(framed(store, witness)) == [{"ada": plain(ada), "bob": plain(bob)}]  # its bindings, by a map over their references

<a id="REF-09"></a>
## REF-09 · A pattern's symbol given a reference in place of a schema: it binds what the reference designates, one frame, or none; its spec filters or is written as any symbol's, by the schema its path implies where the store holds nothing it designates; a symbol has exactly one of a schema and a reference

In [ ]:
reference = References.of(store, ada)  # a symbol given a reference binds what it designates, one frame each
known = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference, "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))
assert plains(known) == [{"x": plain(ada), "y": plain(bob)}, {"x": plain(ada), "y": plain(cy)}]  # from one object, along an adjacency
assert [plain(f["y"]) for f in store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": References.of(store, bob), "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))] == [plain(cy)]
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference}).spec([lambda _: _.x.name("Bob")])]))])) == []  # its spec filters, as any symbol's
upserted = store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": References.of(store, cy)}).spec([lambda _: _.x.name("Cy C.")])]))]))
assert plains(upserted) == [{"x": plain(cy)}] and upserted[0].assigns == {"x": {"name": "Cy C."}}  # an upsert writes its spec over what it binds
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"s": References.of(store, Person)}).spec([lambda _: _.s.name("crm.Person")])]))])) == [{"s": Person}]
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "zed"}).next(lambda n: n.target(lambda t: t))))))
for term in (lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": nowhere})])),
             lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols({"x": nowhere})]))):
    assert store.transaction(lambda t: t.mutations([term])) == []  # it designates nothing: no frame, nothing made
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": nowhere}).spec([lambda _: _.x.name("Zed")])]))])) == []  # its spec, by the schema its path implies


def at(held, singleton, *steps):
    """A `References.OfObject` of `held`'s, from a singleton through entries, each (i, o, keys)."""
    def nested(n, rest):
        if not rest:
            return n.target(lambda t: t)
        (i, o, keys), *more = rest
        return n.entry(lambda e: e.i(i).o(o).properties(keys).next(lambda n: nested(n, more)))
    return held.References.OfObject(lambda o: o.path(lambda p: p.singleton(singleton).next(lambda n: nested(n, steps))))


client = B.Store()  # the schemas, not the data: its spec is written by the schemas its reference's path goes through
client.transaction(crm)
sent = client.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": at(client, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "y": client.registered("crm.Person")})
    .spec([lambda _: _.x.knows([_.y])])]))]))
text = json.dumps(Plain.ToPlain(client).Transaction(sent))  # written as JSON, read where the data is, and run
assert plains(store.transaction(Plain.FromPlain(store).Transaction(json.loads(text)))) == plains(known)  # the frames, as given above


def firm(txn):
    """crm.Firm, owned through crm.Holds as a person is, by an adjacency also named `book`."""
    S = txn.Schemas
    return txn.uid("firm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Firm": S.OfObject}).spec([
            lambda _: _.Holds.name("crm.Holds"),
            lambda _: _.Firm.name("crm.Firm").ref(True).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
        ])]))])


firms = B.Store()
firms.transaction(crm)
firms.transaction(firm)
for held, given, why in (
        (client, at(client, "crm.Nope"), "starts from 'crm.Nope', which no schema declares a singleton"),
        (client, at(client, "crm.TheBook", ("nope", "book", {})),
         "steps through 'nope', an adjacency 'crm.Book' does not declare"),
        (client, at(client, "crm.TheBook", ("contacts", "nope", {})), "steps through 'contacts' of 'crm.Book', where not "
         "exactly one schema declares an adjacency 'nope' marked owner() on another link of its relation: found []"),
        (firms, at(firms, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "steps through 'contacts' of 'crm.Book', "
         "where not exactly one schema declares an adjacency 'book' marked owner() on another link of its relation: "
         "found ['crm.Firm', 'crm.Person']"),
        (client, at(client, ".", ("schemas", "", {"name": "Nope"})), "names a schema object the store does not hold")):
    message = f"symbol 'x' is given a reference whose path {why}: a spec writes what its object's schema declares"
    with raises(LookupError, match=message), held.transaction():
        held.Schemas.OfPattern(lambda p: p.symbols({"x": given}).spec([lambda _: _.x.name("Zed")]))
with store.transaction():
    both = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]))
    neither = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x")]))
    given = store.Schemas.OfPattern(lambda p: p.symbols({"x": reference}))
assert References.validate(both) == ["symbol 'x' has both a schema and a reference: a symbol has exactly one"]
assert References.validate(neither) == ["symbol 'x' has neither a schema nor a reference: a symbol has exactly one"]
assert References.validate(given) == [] and given.symbols["x"].schema is None
with raises(TypeError, match="References.validate takes a pattern, not 'x'"):
    References.validate("x")
with raises(TypeError, match="symbol 'x' has both a schema and a reference: a symbol has exactly one"), \
        store.transaction():
    store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]).spec([
        lambda _: _.x.name("Ada")]))

<a id="REF-10"></a>
## REF-10 · A frame annotated with errors: its reference carries each error's subterm, kind and message, written and read back as any value is

In [ ]:
with store.transaction():
    store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts({"eve": lambda e: e.name("Eve")}))
with store.transaction():  # written where Eve is read: Eve by her path
    eve = contact("eve")
    wire = Plain.ToPlain(store).Transaction(store.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m
        .symbols([lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.name("Ada")])]).then([lambda w: w.update(lambda u: u
        .apply([lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.knows([eve])])]))]))])))
knowing = Plain.FromPlain(store).Transaction(wire)
with store.transaction():
    store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("eve"))))  # Eve deleted
try:
    store.transaction(knowing)
except LookupError as error:
    (annotated,) = error.frames
assert plains([annotated]) == [{"x": plain(ada)}] and len(annotated.errors) == 1
reference = References.of(store, annotated)
assert plain(reference) == {"binds": {"x": plain(References.of(store, ada))}, "assigns": [
    {"object": plain(References.of(store, ada)), "properties": {"name": {"str": "Ada"}}}], "errors": [{
    "subterm": "$.mutations[0].map.then[0].update", "kind": "LookupError",
    "message": "no object of the store has the path a spec refers to"}]}
text = JSON.ToJSON(store)(reference.as_schema(), reference)
assert plain(JSON.FromJSON(store)(reference.as_schema(), text)) == plain(reference)  # carried, the error itself not
assert "errors" not in plain(References.of(store, frame))  # a frame annotated with none: written as before
built = store.References.OfFrame(lambda f: f.binds({"x": lambda o: o.path(at_ada)}).errors([
    lambda x: x.as_object(lambda e: e.subterm("$.mutations[0].upsert").kind("ValueError").message("refused"))]))
assert plain(built)["errors"] == [{"subterm": "$.mutations[0].upsert", "kind": "ValueError", "message": "refused"}]

<a id="REF-11"></a>
## REF-11 · A parameter reference holds its parameter's name and nothing else: a value of the store's, written and read back, which no store resolves

In [ ]:
OfParameter = store.registered("References.OfParameter")
assert list(OfParameter.properties) == ["name"] and OfParameter.properties["name"].schema is store.registered("Schemas.OfStr")
assert not OfParameter.ref  # a value object, as the other references are
n = store.References.OfParameter(lambda p: p.name("n"))  # built by its name: References.of gives none
assert plain(n) == {"name": "n"}
text = JSON.ToJSON(store)(OfParameter, n)
assert text == '{"name": "n"}' and plain(JSON.FromJSON(store)(OfParameter, text)) == {"name": "n"}
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'n'"):
    References.of(store, "n")  # a parameter's name designates no object a store holds